In [ ]:
# Submission path setup: run notebooks from any submission subfolder.
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'Functions.ipynb').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)


In [ ]:
# !pip install "numpy<2.0"
# !pip install torch==2.2.2 torchvision==0.17.2 monai
# !pip install scikit-learn seaborn
# !pip install nibabel nbformat import-ipynb


In [ ]:
import os
import sys
import ast
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import monai
import torch
from tqdm.notebook import tqdm
import nbformat

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from monai.transforms import (
    LoadImaged,
    Compose,
    EnsureChannelFirstd,
    Spacingd,
    ResizeWithPadOrCropd,
    NormalizeIntensityd,
    Lambdad,
    MapTransform,
)


runs_dir = Path("baseline")
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

if not (runs_dir / "best_model.pt").exists():
    raise FileNotFoundError(f"Could not find {(runs_dir / 'best_model.pt')}")

functions_notebook = project_root / "Functions.ipynb"
if not functions_notebook.exists():
    raise FileNotFoundError(f"Could not find {functions_notebook}")


def load_functions_from_notebook(nb_path):
    ns = {}
    nb = nbformat.read(nb_path, as_version=4)
    for cell in nb.cells:
        if cell.cell_type == "code":
            exec(cell.source, ns)
    return ns["patients_dicts"], ns["par_voxelsize"]


patients_dicts, par_voxelsize = load_functions_from_notebook(functions_notebook)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("runs_dir:", runs_dir)
print("project_root:", project_root)
print("device:", device)


In [ ]:
seed = 10
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

checkpoint_path = runs_dir / "best_model.pt"
checkpoint_state = torch.load(checkpoint_path, map_location="cpu")


def infer_channels_from_state_dict(state_dict):
    candidate_keys = [
        "model.0.conv.weight",
        "model.1.submodule.0.conv.weight",
        "model.1.submodule.1.submodule.0.conv.weight",
        "model.1.submodule.1.submodule.1.submodule.0.conv.weight",
        "model.1.submodule.1.submodule.1.submodule.1.submodule.conv.weight",
    ]
    channels = []
    for key in candidate_keys:
        if key in state_dict:
            channels.append(int(state_dict[key].shape[0]))
    if not channels:
        raise RuntimeError("Could not infer channels from checkpoint.")
    return tuple(channels)


summary_path = runs_dir / "summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}

channels = ast.literal_eval(summary["channels"]) if "channels" in summary else infer_channels_from_state_dict(checkpoint_state)
strides = ast.literal_eval(summary["strides"]) if "strides" in summary else (2, 2, 2, 2)
label_map = {1: "RV", 2: "MYO", 3: "LV"}
label_order = ["DCM", "HCM", "MINF", "NOR", "RV"]
disease_label_order = [label for label in label_order if label != "NOR"]

summary_target_spacing = summary.get("target_spacing")
summary_kept_slices = summary.get("kept_slices")
xy_crop_margin_ratio = summary.get("xy_crop_margin_ratio")

print("checkpoint_path:", checkpoint_path)
print("channels:", channels)
print("strides:", strides)
print("target spacing from summary:", summary_target_spacing)
print("kept_slices from summary:", summary_kept_slices)
print("xy_crop_margin_ratio:", xy_crop_margin_ratio)


In [ ]:
def split_dataset(dataset, per_train=16, per_val=4, seed=None):
    if seed is not None:
        random.seed(seed)

    categories = {}
    for patient in dataset:
        disease = patient["Disease"]
        categories.setdefault(disease, []).append(patient)

    split_train_dataset = []
    split_val_dataset = []

    for patient_ids in categories.values():
        random.shuffle(patient_ids)
        split_train_dataset.extend(patient_ids[:per_train])
        split_val_dataset.extend(patient_ids[per_train:per_train + per_val])

    return split_train_dataset, split_val_dataset


class CropForegroundXYd(MapTransform):
    def __init__(self, keys, source_key, margin_ratio=0.35):
        super().__init__(keys)
        self.source_key = source_key
        self.margin_ratio = float(margin_ratio)

    def __call__(self, data):
        d = dict(data)
        source = np.asarray(d[self.source_key])[0]
        foreground = np.where(source != 0)

        if len(foreground[0]) == 0:
            return d

        min_y, max_y = int(foreground[0].min()), int(foreground[0].max())
        min_x, max_x = int(foreground[1].min()), int(foreground[1].max())

        margin_y = int((max_y - min_y + 1) * self.margin_ratio)
        margin_x = int((max_x - min_x + 1) * self.margin_ratio)

        crop_min_y = max(min_y - margin_y, 0)
        crop_max_y = min(max_y + margin_y, source.shape[0] - 1)
        crop_min_x = max(min_x - margin_x, 0)
        crop_max_x = min(max_x + margin_x, source.shape[1] - 1)

        for key in self.keys:
            d[key] = d[key][:, crop_min_y : crop_max_y + 1, crop_min_x : crop_max_x + 1, :]

        return d


full_dict_list = patients_dicts(str(project_root / "train"))
test_dict_list = patients_dicts(str(project_root / "test"))
#using the split function to split the data
train_dict_list, val_dict_list = split_dataset(full_dict_list, per_train=16, per_val=4, seed=seed)

train_dataset_raw = monai.data.Dataset(
    full_dict_list,
    transform=Compose([
        LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
        EnsureChannelFirstd(keys=["imgED", "maskED", "imgES", "maskES"], channel_dim="no_channel"),
    ]),
)

mean_voxel, std_voxel, max_voxelsize = par_voxelsize([], train_dataset_raw)
target_spacing = ast.literal_eval(summary_target_spacing) if summary_target_spacing is not None else tuple(float(x) for x in mean_voxel)
kept_slices = ast.literal_eval(summary_kept_slices) if summary_kept_slices is not None else None
xy_crop_margin_ratio = float(xy_crop_margin_ratio) if xy_crop_margin_ratio is not None else None
roi_size = [256, 256, len(kept_slices)] if kept_slices is not None else [256, 256, 16]
voxel_volume = float(target_spacing[0] * target_spacing[1] * target_spacing[2])

print("split train patients:", len(train_dict_list))
print("split val patients:", len(val_dict_list))
print("test patients:", len(test_dict_list))
print("target spacing:", target_spacing)
print("kept_slices:", kept_slices)
print("xy_crop_margin_ratio:", xy_crop_margin_ratio)
print("roi_size:", roi_size)


In [ ]:
def build_eval_transform():
    transforms = [
        LoadImaged(keys=["imgED", "maskED", "imgES", "maskES"], image_only=False),
        EnsureChannelFirstd(keys=["imgED", "maskED", "imgES", "maskES"], channel_dim="no_channel"),
        Spacingd(
            keys=["imgED", "maskED", "imgES", "maskES"],
            pixdim=(target_spacing[0], target_spacing[1], target_spacing[2]),
            mode=("bilinear", "nearest", "bilinear", "nearest"),
            ensure_same_shape=True,
            align_corners=False,
        ),
    ]

    if xy_crop_margin_ratio is not None:
        transforms.extend([
            CropForegroundXYd(keys=["imgED", "maskED"], source_key="imgED", margin_ratio=xy_crop_margin_ratio),
            CropForegroundXYd(keys=["imgES", "maskES"], source_key="imgES", margin_ratio=xy_crop_margin_ratio),
        ])

    transforms.append(
        ResizeWithPadOrCropd(keys=["imgED", "maskED", "imgES", "maskES"], spatial_size=tuple(roi_size)) #added cause the images needed to be devided by 16 for the strides in the Unet image
    )

    if kept_slices is not None:
        transforms.append(
            Lambdad(keys=["imgED", "maskED", "imgES", "maskES"], func=lambda x: x[..., kept_slices])
        )

    transforms.append(NormalizeIntensityd(keys=["imgED", "imgES"], nonzero=True, channel_wise=True))
    return Compose(transforms)


eval_transform = build_eval_transform()


In [ ]:
#Defining the Unet model
def build_model():
    return monai.networks.nets.UNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=4, #since we want to segment 3 heart areas+ back
        channels=channels, #checking if the chanels are correct
        strides=strides,
        #num_res_units=2,
    ).to(device)


def dice_for_class(pred, gt, class_idx):
    pred_bin = pred == class_idx
    gt_bin = gt == class_idx
    denom = pred_bin.sum() + gt_bin.sum()
    if denom == 0:
        return np.nan
    return float(2.0 * (pred_bin & gt_bin).sum() / denom)


def hd95_for_class(pred, gt, class_idx):
    pred_bin = torch.as_tensor((pred == class_idx)[None, None].astype(np.float32), device=device)
    gt_bin = torch.as_tensor((gt == class_idx)[None, None].astype(np.float32), device=device)

    if float(pred_bin.sum().item()) == 0.0 and float(gt_bin.sum().item()) == 0.0:
        return np.nan

    metric = monai.metrics.HausdorffDistanceMetric(
        include_background=True,
        percentile=95,
        reduction="none",
    )
    metric(y_pred=pred_bin, y=gt_bin)
    value = metric.aggregate().detach().cpu().numpy().reshape(-1)[0]
    metric.reset()
    return float(value) if np.isfinite(value) else np.nan


def predict_masks_for_sample(sample, model, inferer):
    with torch.no_grad():
        image_ed = sample["imgED"].unsqueeze(0).float().to(device)
        image_es = sample["imgES"].unsqueeze(0).float().to(device)
        pred_ed = torch.argmax(inferer(image_ed, network=model), dim=1).squeeze().cpu().numpy().astype(int)
        pred_es = torch.argmax(inferer(image_es, network=model), dim=1).squeeze().cpu().numpy().astype(int)
    return pred_ed, pred_es


feature_cols = [
    "rv_ed", "myo_ed", "lv_ed",
    "rv_es", "myo_es", "lv_es",
    "rv_sv", "lv_sv", "rv_ef", "lv_ef",
    "myo_delta", "lv_rv_ratio_ed", "lv_rv_ratio_es",
    "myo_lv_ratio_ed", "myo_lv_ratio_es",
]


def feature_row_from_masks(patient_id, disease, mask_ed, mask_es):
    def volume_from_mask(mask, label):
        return float((mask == label).sum() * voxel_volume)

    def safe_ef(edv, esv):
        if edv <= 0:
            return np.nan
        return float((edv - esv) / edv)

    rv_ed = volume_from_mask(mask_ed, 1)
    myo_ed = volume_from_mask(mask_ed, 2)
    lv_ed = volume_from_mask(mask_ed, 3)
    rv_es = volume_from_mask(mask_es, 1)
    myo_es = volume_from_mask(mask_es, 2)
    lv_es = volume_from_mask(mask_es, 3)

    return {
        "ID": patient_id,
        "Disease": disease,
        "rv_ed": rv_ed,
        "myo_ed": myo_ed,
        "lv_ed": lv_ed,
        "rv_es": rv_es,
        "myo_es": myo_es,
        "lv_es": lv_es,
        "rv_sv": rv_ed - rv_es,
        "lv_sv": lv_ed - lv_es,
        "rv_ef": safe_ef(rv_ed, rv_es),
        "lv_ef": safe_ef(lv_ed, lv_es),
        "myo_delta": myo_ed - myo_es,
        "lv_rv_ratio_ed": float(lv_ed / rv_ed) if rv_ed > 0 else np.nan,
        "lv_rv_ratio_es": float(lv_es / rv_es) if rv_es > 0 else np.nan,
        "myo_lv_ratio_ed": float(myo_ed / lv_ed) if lv_ed > 0 else np.nan,
        "myo_lv_ratio_es": float(myo_es / lv_es) if lv_es > 0 else np.nan,
    }


In [ ]:
output_dir = Path("baseline/evaluation_final_single")
output_dir.mkdir(exist_ok=True)

split_datasets = {
    "train": monai.data.Dataset(train_dict_list, transform=eval_transform),
    "val": monai.data.Dataset(val_dict_list, transform=eval_transform),
    "test": monai.data.Dataset(test_dict_list, transform=eval_transform),
}

model = build_model()
model.load_state_dict(torch.load(runs_dir / "best_model.pt", map_location=device))
model.eval()
inferer = monai.inferers.SlidingWindowInferer(roi_size=roi_size)

patient_rows = []
classifier_rows = []
feature_tables = []
split_feature_rows = {}

for split_name, dataset in split_datasets.items():
    feature_rows = []
    for sample in tqdm(dataset, desc=f"{split_name}", leave=False):
        pred_ed, pred_es = predict_masks_for_sample(sample, model, inferer)
        gt_ed = np.asarray(sample["maskED"]).squeeze().astype(int)
        gt_es = np.asarray(sample["maskES"]).squeeze().astype(int)

        feature_rows.append(feature_row_from_masks(sample["ID"], sample["Disease"], pred_ed, pred_es))

        for phase, pred_mask, gt_mask in [("ED", pred_ed, gt_ed), ("ES", pred_es, gt_es)]:
            for class_idx, class_name in label_map.items():
                patient_rows.append({
                    "split": split_name,
                    "ID": sample["ID"],
                    "Disease": sample["Disease"],
                    "phase": phase,
                    "structure": class_name,
                    "dice": dice_for_class(pred_mask, gt_mask, class_idx),
                    "hd95": hd95_for_class(pred_mask, gt_mask, class_idx),
                })

    split_feature_rows[split_name] = pd.DataFrame(feature_rows)
    split_feature_rows[split_name]["split"] = split_name
    feature_tables.append(split_feature_rows[split_name])

clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, multi_class="auto", class_weight="balanced"),
)

clf_train_df = split_feature_rows["train"].dropna(subset=feature_cols + ["Disease"]).reset_index(drop=True)
clf.fit(clf_train_df[feature_cols], clf_train_df["Disease"])

for split_name in ["train", "val", "test"]:
    split_df = split_feature_rows[split_name].dropna(subset=feature_cols + ["Disease"]).reset_index(drop=True)
    preds = clf.predict(split_df[feature_cols])
    macro_f1 = f1_score(split_df["Disease"], preds, average="macro")
    recalls = recall_score(split_df["Disease"], preds, labels=label_order, average=None, zero_division=0)
    recall_map = {label: float(recall) for label, recall in zip(label_order, recalls)}
    disease_recalls = [recall_map[label] for label in disease_label_order if label in recall_map]

    row = {
        "split": split_name,
        "macro_f1": float(macro_f1),
        "min_disease_recall": float(min(disease_recalls)) if disease_recalls else np.nan,
    }
    for label in label_order:
        row[f"recall_{label}"] = recall_map.get(label, np.nan)
    classifier_rows.append(row)


In [ ]:
patient_df = pd.DataFrame(patient_rows)
classifier_df = pd.DataFrame(classifier_rows)
features_df = pd.concat(feature_tables, ignore_index=True) if feature_tables else pd.DataFrame()

patient_wide_df = (
    patient_df
    .pivot_table(
        index=["split", "ID", "Disease"],
        columns=["phase", "structure"],
        values=["dice", "hd95"],
        aggfunc="first",
    )
    .sort_index(axis=1)
)
patient_wide_df.columns = [f"{metric}_{structure}_{phase}" for metric, phase, structure in patient_wide_df.columns]
patient_wide_df = patient_wide_df.reset_index()

split_summary_df = (
    patient_df
    .groupby(["split", "phase", "structure"], observed=False)
    .agg(
        dice_mean=("dice", "mean"),
        dice_std=("dice", "std"),
        hd95_mean=("hd95", "mean"),
        hd95_std=("hd95", "std"),
    )
    .reset_index()
)

patient_df.to_csv(output_dir / "patient_metrics_long.csv", index=False)
patient_wide_df.to_csv(output_dir / "patient_metrics_wide.csv", index=False)
split_summary_df.to_csv(output_dir / "split_segmentation_summary.csv", index=False)
classifier_df.to_csv(output_dir / "classifier_summary.csv", index=False)
features_df.to_csv(output_dir / "predicted_feature_table.csv", index=False)

display(patient_wide_df.head())
display(split_summary_df)
display(classifier_df)
print("saved to:", output_dir)


In [ ]:
sns.set_theme(style="whitegrid")

split_order = ["train", "val", "test"]
phase_order = ["ED", "ES"]
structure_order = ["RV", "MYO", "LV"]

patient_df["split"] = pd.Categorical(patient_df["split"], categories=split_order, ordered=True)
patient_df["phase"] = pd.Categorical(patient_df["phase"], categories=phase_order, ordered=True)
patient_df["structure"] = pd.Categorical(patient_df["structure"], categories=structure_order, ordered=True)
classifier_df["split"] = pd.Categorical(classifier_df["split"], categories=split_order, ordered=True)


def nice_dice_ylim(values, floor_cap=0.6, pad=0.02):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return (0.0, 1.0)
    lower = max(0.0, min(floor_cap, np.floor((values.min() - pad) * 20) / 20))
    upper = min(1.0, np.ceil((values.max() + pad) * 20) / 20)
    if upper - lower < 0.08:
        lower = max(0.0, upper - 0.08)
    return (lower, upper)


def positive_hd95(values, min_positive=0.1):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    values = values[values > 0]
    if len(values) == 0:
        return np.array([min_positive], dtype=float)
    return values


def tukey_whisker_limits(values, min_positive=None):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if min_positive is not None:
        values = values[values > min_positive]
    if len(values) == 0:
        base = min_positive if min_positive is not None else 0.0
        return base, max(base + 1.0, 1.0)
    q1, q3 = np.quantile(values, [0.25, 0.75])
    iqr = q3 - q1
    if iqr == 0:
        return values.min(), values.max()
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    inlier_values = values[(values >= lower_fence) & (values <= upper_fence)]
    if len(inlier_values) == 0:
        return values.min(), values.max()
    return inlier_values.min(), inlier_values.max()


def dice_box_ylim(values, floor_cap=0.6, pad=0.02):
    lower, upper = tukey_whisker_limits(values)
    return nice_dice_ylim([lower, upper], floor_cap=floor_cap, pad=pad)


def hd95_box_ylim(values, min_positive=0.1):
    lower, upper = tukey_whisker_limits(values, min_positive=min_positive)
    lower = max(lower, min_positive)
    lower = 10 ** np.floor(np.log10(lower))
    upper = 10 ** np.ceil(np.log10(max(lower * 1.01, upper)))
    if lower == upper:
        upper = lower * 10
    return lower, upper


dice_split_plot = (
    patient_df
    .groupby(["split", "phase", "structure"], as_index=False, observed=False)["dice"]
    .mean()
)
hd95_split_plot = (
    patient_df
    .groupby(["split", "phase", "structure"], as_index=False, observed=False)["hd95"]
    .mean()
)

dice_ylim_all = dice_box_ylim(patient_df["dice"])
hd95_box_ylim_all = hd95_box_ylim(patient_df["hd95"])
classifier_ylim = nice_dice_ylim(
    pd.concat([classifier_df["macro_f1"], classifier_df["min_disease_recall"]], ignore_index=True),
    floor_cap=0.4,
)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

sns.boxplot(data=patient_df, x="split", y="dice", hue="structure", ax=axes[0, 0])
axes[0, 0].set_title("Patient-Level Dice By Split")
axes[0, 0].set_ylim(*dice_ylim_all)

sns.boxplot(data=patient_df[patient_df["hd95"] > 0], x="split", y="hd95", hue="structure", ax=axes[0, 1])
axes[0, 1].set_title("Patient-Level HD95 By Split")
axes[0, 1].set_yscale("log")
axes[0, 1].set_ylim(*hd95_box_ylim_all)

sns.barplot(data=dice_split_plot[dice_split_plot["phase"] == "ED"], x="structure", y="dice", hue="split", errorbar="sd", ax=axes[1, 0])
axes[1, 0].set_title("Mean Dice | ED")
axes[1, 0].set_ylim(*dice_ylim_all)

sns.barplot(data=dice_split_plot[dice_split_plot["phase"] == "ES"], x="structure", y="dice", hue="split", errorbar="sd", ax=axes[1, 1])
axes[1, 1].set_title("Mean Dice | ES")
axes[1, 1].set_ylim(*dice_ylim_all)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

dice_summary = (
    patient_df
    .groupby(["split", "phase", "structure"], observed=False)
    .agg(mean=("dice", "mean"), std=("dice", "std"))
    .reset_index()
)
hd95_summary = (
    patient_df
    .groupby(["split", "phase", "structure"], observed=False)
    .agg(mean=("hd95", "mean"), std=("hd95", "std"))
    .reset_index()
)

for ax, split in zip(axes[0], split_order):
    heat = dice_summary[dice_summary["split"] == split].pivot(index="phase", columns="structure", values="mean")
    sns.heatmap(heat, annot=True, fmt=".3f", cmap="YlGnBu", vmin=0, vmax=1, ax=ax)
    ax.set_title(f"Mean Dice | {split}")

for ax, split in zip(axes[1], split_order):
    heat = hd95_summary[hd95_summary["split"] == split].pivot(index="phase", columns="structure", values="mean")
    sns.heatmap(heat, annot=True, fmt=".2f", cmap="magma_r", ax=ax)
    ax.set_title(f"Mean HD95 | {split}")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=classifier_df, x="split", y="macro_f1", ax=axes[0], errorbar="sd")
axes[0].set_title("Classifier Macro F1")
axes[0].set_ylim(*classifier_ylim)

sns.barplot(data=classifier_df, x="split", y="min_disease_recall", ax=axes[1], errorbar="sd")
axes[1].set_title("Classifier Min Disease Recall")
axes[1].set_ylim(*classifier_ylim)

plt.tight_layout()
plt.show()
